In [3]:
# ============================================================
# CELL 7 — score.py
# PD → Credit Score using log-odds scaling (PDO=20, Base=800, Odds=50)
# Bands: A1 (750–900) → A2 → B1 → B2 → C → D (200–549)
#
# Uses   →  master  (DataFrame from Cell 6)
# Reads  →  models/gbm_model.pkl
#            models/scaler.pkl
#            models/feature_list.json
#            config/scoring_cutoff.yaml
# Produces → scored  (master + pd, score, band, risk_category, recommendation)
# ============================================================

import json
import math
import yaml
import joblib
import numpy as np
import pandas as pd

# ── 1. Load artifacts (once, at cell run) ────────────────────
print("=" * 60)
print("  STEP 7 — PRODUCTION SCORING ENGINE")
print("=" * 60)

model    = joblib.load("D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models/gbm_model.pkl")
scaler   = joblib.load("D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models/scaler.pkl")

with open("D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models/feature_list.json")   as _f: features = json.load(_f)
with open("D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/config/scoring_cutoff.yaml") as _f: cutoffs  = yaml.safe_load(_f)

_cfg = cutoffs["pd_to_score"]
BASE_SCORE = _cfg["base_score"]   # 800
PDO        = _cfg["pdo"]          # 20  — points to double odds
BASE_ODDS  = _cfg["base_odds"]    # 50  — good:bad odds at base score

print(f"  Model loaded   : {type(model).__name__}")
print(f"  Features       : {len(features)}")
print(f"  Score bands    : {len(cutoffs['score_bands'])} (A1 → D)")
print(f"  PDO params     : Base={BASE_SCORE}, PDO={PDO}, Odds={BASE_ODDS}")


# ── 2. Core formula: PD → Credit Score ───────────────────────
# Log-odds scaling — same method used by CIBIL / Experian.
# Higher PD = lower score. Clamped to [200, 900].
#
#   score = base + (PDO / ln2) × (ln(odds) + ln((1−pd)/pd))
#
# Intuition:
#   PD=1%  →  score ≈ 820  (very safe)
#   PD=12% →  score ≈ 700  (moderate)
#   PD=50% →  score ≈ 627  (risky — 1:1 odds)
#   PD=80% →  score ≈ 378  (very high risk)

def pd_to_score(pd_val: float) -> float:
    pd_val = max(1e-6, min(1 - 1e-6, pd_val))          # guard: avoid log(0)
    log_odds = math.log((1 - pd_val) / pd_val)          # log of good:bad odds
    score    = BASE_SCORE + (PDO / math.log(2)) * (math.log(BASE_ODDS) + log_odds)
    return round(max(200.0, min(900.0, score)), 1)


# ── 3. Band lookup from scoring_cutoff.yaml ───────────────────
_BANDS = cutoffs["score_bands"]   # list of band dicts, already in order

def get_score_band(score: float) -> dict:
    for b in _BANDS:
        if b["min_score"] <= score <= b["max_score"]:
            return b
    return {                                             # fallback (shouldn't hit)
        "band": "D", "risk_category": "Very High Risk",
        "recommended_action": "Decline",
        "max_ltv": 0.0, "max_foir": 0.0,
    }


# ── 4. Score a single applicant record (dict → dict) ─────────
def score_record(record: dict) -> dict:
    """
    Pass any dict of feature values.
    Missing features are filled with 0 automatically.
    Returns pd, score, band, risk_category, recommendation.
    """
    df     = pd.DataFrame([record]).reindex(columns=features, fill_value=0)
    pd_val = float(model.predict_proba(df.values)[0][1])
    sc     = pd_to_score(pd_val)
    band   = get_score_band(sc)
    return {
        "pd":             round(pd_val, 6),
        "score":          sc,
        "band":           band["band"],
        "risk_category":  band["risk_category"],
        "recommendation": band["recommended_action"],
        "max_ltv":        band["max_ltv"],
        "max_foir":       band["max_foir"],
    }


# ── 5. Score entire DataFrame (vectorised) ───────────────────
def score_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Scores every row of df.
    Adds columns: pd, score, band, risk_category, recommendation, max_ltv, max_foir.
    """
    X    = df.reindex(columns=features, fill_value=0).values
    pds  = model.predict_proba(X)[:, 1]                # shape (n,)

    out                    = df.copy()
    out["pd"]              = np.round(pds, 6)
    out["score"]           = [pd_to_score(p) for p in pds]

    _band_rows             = [get_score_band(s) for s in out["score"]]
    out["band"]            = [b["band"]               for b in _band_rows]
    out["risk_category"]   = [b["risk_category"]      for b in _band_rows]
    out["recommendation"]  = [b["recommended_action"] for b in _band_rows]
    out["max_ltv"]         = [b["max_ltv"]             for b in _band_rows]
    out["max_foir"]        = [b["max_foir"]            for b in _band_rows]
    return out


# ── 6. Score the master dataset ───────────────────────────────
try:
    master
except NameError:
    master = pd.read_csv(
        "D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/data/processed/model_dataset.csv"
    )
    print(f"  master loaded from disk: {master.shape[0]:,} rows × {master.shape[1]} cols")

scored = score_dataframe(master)

# ── 7. Output ─────────────────────────────────────────────────
print(f"\n  Scored records : {len(scored):,}")

# Band distribution
_band_dist = (
    scored.groupby("band")
    .agg(count=("loan_id", "count"), avg_pd=("pd", "mean"), avg_score=("score", "mean"))
    .reset_index()
    .assign(
        pct        = lambda x: (x["count"] / len(scored) * 100).round(1),
        avg_pd     = lambda x: x["avg_pd"].round(4),
        avg_score  = lambda x: x["avg_score"].round(1),
    )
    [["band", "count", "pct", "avg_score", "avg_pd"]]
)

_band_order = ["A1", "A2", "B1", "B2", "C", "D"]
_band_dist["band"] = pd.Categorical(_band_dist["band"], categories=_band_order, ordered=True)
_band_dist = _band_dist.sort_values("band")

print("\n  Band Distribution:")
print("-" * 60)
print(_band_dist.to_string(index=False))

# Approval summary
_approve_bands = {"A1", "A2", "B1"}
_auto_approve  = scored[scored["band"] == "A1"]
_approve       = scored[scored["band"].isin(_approve_bands)]
_decline       = scored[scored["band"] == "D"]

print(f"\n  Auto-Approve (A1)          : {len(_auto_approve):>4} ({len(_auto_approve)/len(scored)*100:.1f}%)")
print(f"  Approve incl. scrutiny (≤B1): {len(_approve):>4} ({len(_approve)/len(scored)*100:.1f}%)")
print(f"  Decline (D)                : {len(_decline):>4} ({len(_decline)/len(scored)*100:.1f}%)")

# Score stats
print(f"\n  Score range  : {scored['score'].min():.1f} – {scored['score'].max():.1f}")
print(f"  Score mean   : {scored['score'].mean():.1f}")
print(f"  PD range     : {scored['pd'].min():.4f} – {scored['pd'].max():.4f}")

# Sample rows
print(f"\n  Sample output:")
print("-" * 60)
_show = ["loan_id", "pd", "score", "band", "risk_category", "recommendation", "max_ltv", "max_foir"]
_show = [c for c in _show if c in scored.columns]
print(scored[_show].head(10).to_string(index=False))
print("=" * 60)


  STEP 7 — PRODUCTION SCORING ENGINE
  Model loaded   : GradientBoostingClassifier
  Features       : 86
  Score bands    : 6 (A1 → D)
  PDO params     : Base=800, PDO=20, Odds=50
  master loaded from disk: 300 rows × 111 cols

  Scored records : 300

  Band Distribution:
------------------------------------------------------------
band  count   pct  avg_score  avg_pd
  A1    300 100.0      880.1  0.5235

  Auto-Approve (A1)          :  300 (100.0%)
  Approve incl. scrutiny (≤B1):  300 (100.0%)
  Decline (D)                :    0 (0.0%)

  Score range  : 818.7 – 900.0
  Score mean   : 880.1
  PD range     : 0.0519 – 0.9632

  Sample output:
------------------------------------------------------------
 loan_id       pd  score band risk_category recommendation  max_ltv  max_foir
 1000001 0.728122  884.5   A1 Very Low Risk   Auto Approve      0.9      0.65
 1000002 0.134958  900.0   A1 Very Low Risk   Auto Approve      0.9      0.65
 1000005 0.936230  835.4   A1 Very Low Risk   Auto Appro